# Persian human annotation pilot: data and analysis

32 pilot items, three annotators, four scenarios. This notebook does not use the separate candidate set. [Archived release](https://doi.org/10.5281/zenodo.24331046).

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'scripts/analyze.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the repository or its notebooks/ folder.')
print(ROOT.name)

## Check the factorial coverage and annotation records

In [ ]:
DATA = ROOT / 'data'
if not (DATA / 'annotations.csv').is_file():
    DATA = DATA / 'pilot'
annotations = pd.read_csv(DATA / 'annotations.csv')
stimuli = pd.read_csv(DATA / 'stimuli.csv')
assert len(annotations) == 96 and len(stimuli) == 32
assert annotations['annotator_id'].nunique() == 3
assert annotations['scenario_id'].nunique() == 4
assert not annotations.duplicated(['public_item_id', 'annotator_id']).any()
assert set(annotations['public_item_id']) == set(stimuli['public_item_id'])
assert annotations.groupby('public_item_id')['annotator_id'].nunique().eq(3).all()
factors = ['policy_stance_intended', 'hostility_intended', 'arousal_intended']
coverage = pd.concat([stimuli.groupby(factors).size().rename('Stimuli'),
                      annotations.groupby(factors).size().rename('Annotation records')], axis=1)
assert len(coverage) == 8 and coverage['Stimuli'].eq(4).all()
display(coverage)

## Recompute results

The notebook calls the repository analysis scripts in a temporary workspace, so committed files are not overwritten. The workspace also accommodates the scripts’ `data/pilot/` input paths.

In [ ]:
import shutil, tempfile
with tempfile.TemporaryDirectory() as tmp:
    work = Path(tmp)
    shutil.copytree(ROOT / 'scripts', work / 'scripts')
    (work / 'data/pilot').mkdir(parents=True)
    for name in ('annotations.csv', 'stimuli.csv'):
        shutil.copy2(DATA / name, work / 'data/pilot' / name)
    for script in ('validate.py', 'analyze.py'):
        run = subprocess.run([sys.executable, f'scripts/{script}'], cwd=work,
                             check=True, capture_output=True, text=True)
        print(run.stdout)
    tables = {name: pd.read_csv(work / 'results' / f'{name}.csv') for name in (
        'descriptives', 'matched_tests', 'pairwise_agreement', 'stance_confusion',
        'hostility_expression_counts', 'group_target')}
    import json
    summary = json.loads((work / 'results/summary.json').read_text())

In [ ]:
display(tables['descriptives'].round(3))
columns = ['comparison','n_pairs','nonzero_pairs','mean_difference','sd_difference','W','reporting_method','reporting_p']
display(tables['matched_tests'][columns].round(5))
for name in ('pairwise_agreement', 'stance_confusion', 'hostility_expression_counts', 'group_target'):
    print(name.replace('_', ' ').title())
    display(tables[name].round(3))
print('ICC:', summary['reliability'])

**Interpretation:** The matched comparisons are exploratory. Contrasts share four scenarios, and the two arousal–hostility comparisons do not constitute an interaction test.